In [3]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0, ResNet50V2, VGG16, MobileNetV2
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.resnet_v2 import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization, Flatten
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import regularizers
from sklearn.metrics import accuracy_score, f1_score
from collections import Counter
import gc

# --- Configuración ---
DATASET_PATH = '/kaggle/input/psoriasis-ds/dataset_v9_v4'
IMG_SIZE = 224
BATCH_SIZE = 16
EPOCHS_KFOLD = 10
RANDOM_SEED = 44
N_SPLITS = 5

# --- Modelos a Comparar ---
MODELS_TO_COMPARE = {
    'VGG16': {'model': VGG16, 'preprocess': vgg_preprocess},
    'ResNet50V2': {'model': ResNet50V2, 'preprocess': resnet_preprocess},
    'MobileNetV2': {'model': MobileNetV2, 'preprocess': mobilenet_preprocess},
    'EfficientNetB0': {'model': EfficientNetB0, 'preprocess': efficientnet_preprocess}
}

# --- Funciones de Utilidad ---

def load_all_images(directory, img_size):
    """Carga todas las imágenes del directorio y devuelve arrays de imágenes y etiquetas."""
    classes = sorted(os.listdir(directory))
    images, labels = [], []
    class_indices = {name: i for i, name in enumerate(classes)}
    for class_name in classes:
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path): continue
        for img_name in os.listdir(class_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                img_path = os.path.join(class_path, img_name)
                try:
                    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(img_size, img_size))
                    img_array = tf.keras.preprocessing.image.img_to_array(img)
                    images.append(img_array)
                    labels.append(class_indices[class_name])
                except Exception as e:
                    print(f"Error cargando {img_path}: {e}")
    return np.array(images), np.array(labels), classes

def create_classifier_head(base_model, num_classes):
    """Crea el clasificador personalizado sobre la base congelada."""
    # VGG16 no suele usar GAP, sino Flatten
    if 'vgg16' in base_model.name:
        x = Flatten()(base_model.output)
    else:
        x = GlobalAveragePooling2D()(base_model.output)
        
    x = Dense(512, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    x = Dense(256, activation='relu', kernel_regularizer=regularizers.l2(0.001))(x)
    x = Dropout(0.2)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base_model.input, outputs=outputs)

def balance_dataset(X_data, y_data, num_classes):
    """Balancea el dataset de entrenamiento."""
    datagen = ImageDataGenerator(rotation_range=30, width_shift_range=0.2, height_shift_range=0.2,
                                 shear_range=0.2, zoom_range=0.3, horizontal_flip=True, fill_mode='nearest')
    class_counts = Counter(y_data)
    if not class_counts: return np.array([]), None
    max_count = max(class_counts.values())
    target_count = int(max_count * 2.5)
    
    X_balanced, y_balanced = list(X_data), list(y_data)
    for class_idx in range(num_classes):
        current_count = class_counts.get(class_idx, 0)
        if 0 < current_count < target_count:
            needed = target_count - current_count
            images_to_augment = X_data[y_data == class_idx]
            augmented_imgs = []
            reps = int(np.ceil(needed / len(images_to_augment))) if len(images_to_augment) > 0 else 0
            for _ in range(reps):
                if len(augmented_imgs) >= needed: break
                for img_orig in images_to_augment:
                    if len(augmented_imgs) >= needed: break
                    img = np.expand_dims(img_orig, 0)
                    aug_batch = next(datagen.flow(img, batch_size=1))
                    augmented_imgs.append(aug_batch[0])
            X_balanced.extend(augmented_imgs)
            y_balanced.extend([class_idx] * len(augmented_imgs))
            
    return np.array(X_balanced), tf.keras.utils.to_categorical(np.array(y_balanced), num_classes=num_classes)

# --- Pipeline Principal de Comparación ---

results_summary = []

# Cargar datos una sola vez
print("Cargando imágenes (sin preprocesar)...")
all_images_raw, y_all, class_names = load_all_images(DATASET_PATH, IMG_SIZE)
num_classes = len(class_names)

for model_name, model_info in MODELS_TO_COMPARE.items():
    print(f"\n{'='*20} EVALUANDO ARQUITECTURA: {model_name} {'='*20}")
    
    # Preprocesar datos para el modelo actual
    print(f"Preprocesando datos para {model_name}...")
    X_all_preprocessed = model_info['preprocess'](all_images_raw.copy())
    
    kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    fold_accuracies, fold_f1_macros = [], []

    for fold_idx, (train_indices, val_indices) in enumerate(kf.split(X_all_preprocessed, y_all)):
        print(f"\n--- Fold {fold_idx + 1}/{N_SPLITS} para {model_name} ---")
        X_train, X_val = X_all_preprocessed[train_indices], X_all_preprocessed[val_indices]
        y_train, y_val = y_all[train_indices], y_all[val_indices]
        
        X_train_balanced, y_train_balanced_cat = balance_dataset(X_train, y_train, num_classes)
        y_val_cat = tf.keras.utils.to_categorical(y_val, num_classes=len(class_names))
        
        # Crear modelo
        base_model = model_info['model'](weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
        base_model.trainable = False
        model = create_classifier_head(base_model, len(class_names))
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='categorical_crossentropy', metrics=['accuracy'])
        
        callbacks = [
            EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=0),
            ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=0)
        ]
        
        model.fit(X_train_balanced, y_train_balanced_cat, validation_data=(X_val, y_val_cat),
                  epochs=EPOCHS_KFOLD, batch_size=BATCH_SIZE, callbacks=callbacks, verbose=0)
        
        # Evaluar
        y_pred_probs = model.predict(X_val, verbose=0)
        y_pred = np.argmax(y_pred_probs, axis=1)
        
        acc = accuracy_score(y_val, y_pred)
        f1 = f1_score(y_val, y_pred, average='macro', zero_division=0)
        fold_accuracies.append(acc)
        fold_f1_macros.append(f1)
        
        print(f"Fold {fold_idx + 1} -> Accuracy: {acc:.4f}, F1-Macro: {f1:.4f}")
        
        del model, base_model
        tf.keras.backend.clear_session()
        gc.collect()

    # Guardar resultados del modelo
    results_summary.append({
        'Arquitectura': model_name,
        'Accuracy Promedio': np.mean(fold_accuracies),
        'Accuracy Std': np.std(fold_accuracies),
        'F1-Macro Promedio': np.mean(fold_f1_macros),
        'F1-Macro Std': np.std(fold_f1_macros)
    })

# --- Imprimir Tabla de Resumen Final ---
print("\n\n" + "="*50)
print("RESUMEN COMPARATIVO DE ARQUITECTURAS (K-Fold CV)")
print("="*50)
df_summary = pd.DataFrame(results_summary)
print(df_summary.to_string(index=False, float_format="%.4f"))


Cargando imágenes (sin preprocesar)...

==================== EVALUANDO ARQUITECTURA: VGG16 ====================
Preprocesando datos para VGG16...

--- Fold 1/5 para VGG16 ---
Fold 1 -> Accuracy: 0.8875, F1-Macro: 0.8722

--- Fold 2/5 para VGG16 ---
Fold 2 -> Accuracy: 0.9000, F1-Macro: 0.9137

--- Fold 3/5 para VGG16 ---
Fold 3 -> Accuracy: 0.8375, F1-Macro: 0.8310

--- Fold 4/5 para VGG16 ---
Fold 4 -> Accuracy: 0.8742, F1-Macro: 0.8748

--- Fold 5/5 para VGG16 ---
Fold 5 -> Accuracy: 0.8616, F1-Macro: 0.8676

==================== EVALUANDO ARQUITECTURA: ResNet50V2 ====================
Preprocesando datos para ResNet50V2...

--- Fold 1/5 para ResNet50V2 ---
94668760/94668760 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Fold 1 -> Accuracy: 0.8625, F1-Macro: 0.8632

--- Fold 2/5 para ResNet50V2 ---
Fold 2 -> Accuracy: 0.8750, F1-Macro: 0.8703

--- Fold 3/5 para ResNet50V2 ---
Fold 3 -> Accuracy: 0.8625, F1-Macro: 0.8472

--- Fold 4/5 para ResNet50V2 ---
Fold 4 -> Accuracy: 0.8868, F1-Macro: 0.8840
